# Interactive smoke test — model load, forward, plots, errors

Notebook companion to `smoke_test_trainval_rf.py`. Loads a model, picks one sample from the test set, runs forward at a chosen input size and (optional) rotation/flip, and visualises ground truth, reconstruction, latent, and per-channel error metrics.

Set everything in the **Config** cell, then re-run from there. The notebook imports helpers (`_apply_transform`, `_invert_transform`, `_to_plain`, `_crop_to_input`, `_config_with_overrides`) from `smoke_test_trainval_rf.py` so behaviour stays in sync with the smoke test.

FORWARD HERE IS UNMASKED. The encoder sees every channel and the decoder reconstructs all of them — same regime as `_forward` in `evaluate_equivariance_v3.py`. If you want the masked training-val regime instead, copy the masking block from `smoke_test_trainval_rf.py`.

## Imports

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from load_model_for_evaluation import load_model_and_data
from smoke_test_trainval_rf import (
    _config_with_overrides,
    _apply_transform,
    _invert_transform,
    _to_plain,
    _crop_to_input,
)

## Config — set these and re-run the cells below

In [ ]:
CONFIG_PATH = 'train_masked_equivariant_config_flip_v2_wider.yaml'
CHECKPOINT_PATH = 'checkpoints/checkpoint-EquivariantConvnext_v2_20260403_101452_J2506143-epoch_199.pth'

# Optional overrides — leave at None to use the config's values.
MODEL_TYPE = None        # e.g. 'EquivariantConvnextV2' to force the v2 module
INPUT_SIZE = None        # e.g. 128 to bypass the [3:-4] decoder crop

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Which sample to look at.
BATCH_INDEX = 0          # which test-dataloader batch (iterated linearly)
SAMPLE_IN_BATCH = 0      # which sample within that batch

# Transform to probe. 'identity' = no rotation; otherwise the reconstruction
# and latent for T(x) are computed and brought back via T^-1 for comparison.
TRANSFORM = 'identity'   # 'identity' | 'rot90' | 'rot180' | 'rot270' | 'hflip' | 'vflip'

# Visualisation knobs.
CHANNELS_TO_PLOT = [0, 1, 2, 3]   # channel positions to show in the image grids
NUM_LATENT_CHANNELS = 6           # top-K by L2 norm of E(x)
PLOT_NCOLS = 4                    # grid width

## Load model + dataloader

Applies any `--model-type` / `--input-size` overrides via a temp config so the shared config file isn't modified.

In [ ]:
cfg_path, tmp_cfg = _config_with_overrides(
    CONFIG_PATH, model_type=MODEL_TYPE, input_size=INPUT_SIZE,
)
try:
    model, _, test_dataloader, _tok, inv_tokenizer, config = load_model_and_data(
        config_path=cfg_path, checkpoint_path=CHECKPOINT_PATH, device=DEVICE,
    )
finally:
    if tmp_cfg and os.path.exists(tmp_cfg):
        os.unlink(tmp_cfg)
model.eval()
print(f'effective input_image_size = {config["input_image_size"]}')
print(f'num test batches: {len(test_dataloader)}')

## Pick a sample

Iterates the test_dataloader up to `BATCH_INDEX` (linear scan — the sampler isn't indexable). Each batch comes from a single panel; the channel set depends on `BATCH_INDEX`.

In [ ]:
for i, batch in enumerate(test_dataloader):
    if i == BATCH_INDEX:
        img_b, channel_ids_b, _panel_idx, _img_path = batch
        break

img_b = img_b.to(DEVICE).float()
channel_ids_b = channel_ids_b.to(DEVICE).long()

img = img_b[SAMPLE_IN_BATCH:SAMPLE_IN_BATCH + 1]                  # (1, C, H, W)
channel_ids = channel_ids_b[SAMPLE_IN_BATCH:SAMPLE_IN_BATCH + 1]  # (1, C)

print(f'sample shape: {tuple(img.shape)}')
print(f'num channels in panel: {channel_ids.shape[1]}')
print(f'markers: {[inv_tokenizer.get(int(c), str(int(c))) for c in channel_ids[0]]}')

## Forward — reference (and transformed if `TRANSFORM != "identity"`)

`recon_ref = D(E(x))`, `latent_ref = E(x)`. When a transform is requested we also compute `recon_T_back = T^-1(D(E(T(x))))` and `latent_T_back = T^-1(E(T(x)))`.

In [ ]:
@torch.no_grad()
def forward_full(model, img, channel_ids):
    """All channels in, all channels out. Returns (recon mean, latent)."""
    out = model(img, channel_ids, channel_ids, True)
    output = _crop_to_input(out['output'], img.shape[-2])
    mi = torch.sigmoid(output[..., 0]).float()
    latent = _to_plain(out['features'][-1]).float()
    return mi, latent

recon_ref, lat_ref = forward_full(model, img, channel_ids)

if TRANSFORM != 'identity':
    img_T = _apply_transform(img, TRANSFORM)
    recon_T, lat_T = forward_full(model, img_T, channel_ids)
    recon_T_back = _invert_transform(recon_T, TRANSFORM)
    lat_T_back = _invert_transform(lat_T, TRANSFORM)
else:
    img_T = recon_T_back = lat_T_back = None

print(f'recon  shape: {tuple(recon_ref.shape)}')
print(f'latent shape: {tuple(lat_ref.shape)}')
print(f'TRANSFORM = {TRANSFORM!r}')

## Plot helper

In [ ]:
def plot_grid(arrays, titles, cmap='magma', vmin=None, vmax=None, ncols=None, suptitle=None):
    """Lay out a list of 2-D arrays as a small image grid."""
    n = len(arrays)
    ncols = min(ncols or PLOT_NCOLS, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows), squeeze=False)
    for i, (arr, t) in enumerate(zip(arrays, titles)):
        ax = axes[i // ncols][i % ncols]
        vmin_ = 0.0 if vmin is None else vmin
        vmax_ = float(np.abs(arr).max()) if vmax is None else vmax
        if vmax_ <= 0:
            vmax_ = 1e-6
        ax.imshow(arr, cmap=cmap, vmin=vmin_, vmax=vmax_)
        ax.set_title(t, fontsize=9)
        ax.axis('off')
    for j in range(n, nrows * ncols):
        axes[j // ncols][j % ncols].axis('off')
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.96 if suptitle else 1))
    return fig

## Ground truth

In [ ]:
channels_to_plot = [c for c in CHANNELS_TO_PLOT if 0 <= c < img.shape[1]]
names = [inv_tokenizer.get(int(channel_ids[0, c]), f'ch{c}') for c in channels_to_plot]

gts = [img[0, c].cpu().numpy() for c in channels_to_plot]
plot_grid(gts, [f'GT — {n}' for n in names], cmap='magma', vmax=1.0, suptitle='Ground truth')
plt.show()

## Reconstruction (reference, D(E(x)))

In [ ]:
recs = [recon_ref[0, c].cpu().numpy() for c in channels_to_plot]
plot_grid(recs, [f'Recon — {n}' for n in names], cmap='magma', vmax=1.0, suptitle='Reconstruction')
plt.show()

## Reconstruction under transform — T⁻¹(D(E(T(x))))  (only if `TRANSFORM != identity`)

In [ ]:
if recon_T_back is not None:
    recs_back = [recon_T_back[0, c].cpu().numpy() for c in channels_to_plot]
    plot_grid(recs_back, [f'T⁻¹∘D∘E∘T — {n}' for n in names],
              cmap='magma', vmax=1.0,
              suptitle=f'T = {TRANSFORM}, brought back to reference frame')
    plt.show()
else:
    print('TRANSFORM is identity — skipping.')

## Latent

Channels picked by L2 norm of `E(x)` (most active first). RdBu colormap centred at 0 since latent values can be signed.

In [ ]:
lat_ref_np = lat_ref[0].cpu().numpy()
norms = np.sqrt((lat_ref_np ** 2).reshape(lat_ref_np.shape[0], -1).sum(axis=1))
k = min(NUM_LATENT_CHANNELS, lat_ref_np.shape[0])
picked = np.argsort(-norms)[:k].tolist()
print(f'latent shape (C, H, W) = {lat_ref_np.shape}')
print(f'picked latent channels (by L2 norm of E(x)): {picked}')

vmax_lat = float(np.abs(lat_ref_np[picked]).max())
plot_grid(
    [lat_ref_np[c] for c in picked],
    [f'E(x) ch{c}' for c in picked],
    cmap='RdBu_r', vmin=-vmax_lat, vmax=vmax_lat,
    suptitle='Latent E(x) — selected channels',
)
plt.show()

## Latent under transform — T⁻¹(E(T(x))) and |diff|  (only if `TRANSFORM != identity`)

If the encoder is perfectly equivariant, `T⁻¹(E(T(x))) == E(x)` and `|diff|` is at machine precision.

In [ ]:
if lat_T_back is not None:
    lat_back_np = lat_T_back[0].cpu().numpy()
    plot_grid(
        [lat_back_np[c] for c in picked],
        [f'T⁻¹∘E∘T ch{c}' for c in picked],
        cmap='RdBu_r', vmin=-vmax_lat, vmax=vmax_lat,
        suptitle=f'Latent under T = {TRANSFORM}, brought back',
    )
    plt.show()

    diff = np.abs(lat_ref_np - lat_back_np)
    plot_grid(
        [diff[c] for c in picked],
        [f'|diff| ch{c}  (max={diff[c].max():.2e})' for c in picked],
        cmap='magma',
        suptitle='Latent equivariance error |E(x) − T⁻¹(E(T(x)))|',
    )
    plt.show()
else:
    print('TRANSFORM is identity — skipping.')

## Errors

All MSEs are computed per channel and then aggregated. `Recon vs GT` is the reconstruction quality on the reference input. When a non-identity transform is used we additionally report consistency (`T⁻¹∘D∘E∘T vs D∘E`), accuracy of the rotated-back reconstruction (`T⁻¹∘D∘E∘T vs x`), and the latent equivariance error.

In [ ]:
def per_channel_mse(a, b):
    """Both inputs (1, C, H, W). Returns (C,) numpy."""
    return ((a[0] - b[0]) ** 2).mean(dim=(-2, -1)).cpu().numpy()

err_recon_vs_gt = per_channel_mse(recon_ref, img)
print(f'Recon vs GT                : mean MSE = {err_recon_vs_gt.mean():.4e},  median = {np.median(err_recon_vs_gt):.4e}')

if recon_T_back is not None:
    err_consistency   = per_channel_mse(recon_T_back, recon_ref)
    err_recon_T_vs_gt = per_channel_mse(recon_T_back, img)
    lat_err = ((lat_ref[0] - lat_T_back[0]) ** 2).mean(dim=(-2, -1)).cpu().numpy()
    print(f'Consistency  T⁻¹∘D∘E∘T vs D∘E : mean MSE = {err_consistency.mean():.4e}')
    print(f'Rotated recon vs GT          : mean MSE = {err_recon_T_vs_gt.mean():.4e}')
    print(f'Latent equivariance T⁻¹∘E∘T vs E : mean MSE = {lat_err.mean():.4e},  max = {lat_err.max():.4e}')

## Per-channel table (selected channels)

In [ ]:
import pandas as pd

row_idx = channels_to_plot
data = {
    'channel_pos': row_idx,
    'marker': names,
    'mse_recon_vs_gt': [float(err_recon_vs_gt[c]) for c in row_idx],
}
if recon_T_back is not None:
    err_consistency_full   = per_channel_mse(recon_T_back, recon_ref)
    err_recon_T_vs_gt_full = per_channel_mse(recon_T_back, img)
    data['mse_consistency']      = [float(err_consistency_full[c]) for c in row_idx]
    data['mse_rot_recon_vs_gt']  = [float(err_recon_T_vs_gt_full[c]) for c in row_idx]
df = pd.DataFrame(data)
df.style.format({c: '{:.4e}' for c in df.columns if c.startswith('mse_')})